# Phase 7 (Prior Case Retrieval) — Run #1: Phase-1 **Decoder-only model 02** (hybrid local/global attention + GQA) as a **bi-encoder** with **LoRMA**

Config-driven like the Phase-2…6 notebooks: edit **Cell 2** only. The compulsory technique (LoRMA, Bihany, Patel & Modi 2025) on the Dec-02 checkpoint. It is also a **KD-teacher candidate** (`EXPORT_TEACHER=True`): it exports its similarity scores over every train query's BM25 top-300 for the distillation run.

### What the data audit (Step 0) found, and what it decided here
| Audit finding | Consequence in this notebook |
|---|---|
| 827 / 118 / 237 queries; pools of 4,320 / 1,023 / 1,727; **every query is inside its own pool** | the query is **removed from its own ranking** everywhere (training negatives, dev, test) |
| 95 % of the train pool is gold for some query; 37 % of gold pairs involve a candidate that is gold for ≥ 2 queries | in-batch negatives are **masked** whenever they are gold for that query (false-negative masking) |
| gold/query median 4–5, mean 6.4–8.2 → ceiling **F1@7 = 0.618** on dev | scores are read against that ceiling, not against 1.0 |
| tokens/doc median ~5.6k, p90 ~19k; 16 × 512 reads 86 % of all tokens | candidates: up to `MAX_DOC_CHUNKS = 16` chunks of 512; training samples `C_TRAIN` chunks per document |
| `<CITATION>` in 99.9 % of queries, ~10 per query, corr. 0.79 with the gold count, spread over the whole document; `<ORG>` / `<NAME>` anonymisation | query **evidence view**: the sentences around each `<CITATION>` (the query's own description of why it cites a case) + the head of the judgment |
| year proxy: only 81 % of gold pairs satisfy `year(gold) ≤ year(query)`, BM25 gets worse with the filter | **no year filter** |
| BM25 (uni+bigram, k1 = 2.2, b = 1.0, log query-TF) dev **F1@7 = 0.388** (63 % of the ceiling), MRR 0.83, recall@100 0.69, recall@300 0.83 | BM25 is the **fusion partner** and the source of **hard negatives**; the run is judged on how much it adds to BM25 |

### The model
* **Encoder.** The Phase-1 Dec-02 backbone, LoRMA on all 7 projections of every block (only the LoRMA factors and a small head are trained). A chunk is `[BOS] + ≤510 tokens + [EOS]`, right-padded. Right padding needs no attention mask in a causal model: a real token at position `t` only attends to positions `≤ t`, which are all real.
* **Pooling.** A causal LM's hidden state `h_t` has only seen tokens `1..t`, so plain mean pooling over-weights early positions that know little about the chunk. Three options are compared **zero-shot** (before any training, this is the "how do Phase-1 representations transfer?" measurement) and the best one is used: `mean`; `wmean`, the position-weighted mean of SGPT (Muennighoff 2022), `e = Σ_t t·h_t / Σ_t t`, which up-weights late, well-informed positions; and `last`, the `[EOS]` state, the only position that has attended to the whole chunk (in Dec-02 through the 3 global layers; the 9 local layers see 256 tokens).
* **Head.** A `d × d` linear map initialised to the **identity**, so step 0 is exactly the Phase-1 model; LoRMA+ also starts at identity (`B = 0`).
* **Document score.** Each document is a set of normalised chunk vectors. `mean`: cosine between the normalised means. `maxsim`: `s(q,d) = (1/|q|) Σ_i max_j cos(q_i, d_j)` — each query chunk is matched to its best candidate chunk (FirstP/MaxP, Dai & Callan 2019), which suits long judgments where only a few passages are relevant. Both are evaluated; the better one on dev is used.

### The objective: multi-positive InfoNCE with hard negatives
A batch has `B` (query, positive) pairs and one BM25 hard negative per pair, so `2B` documents. For query `i` with positive column `p(i)`:

`L_i = −log [ exp(s_{i,p(i)}/τ) / Σ_{j ∉ M_i} exp(s_{ij}/τ) ]`,

where `M_i` = all *other* columns that are gold for `i` (or are `i` itself). Without the mask, a document that is gold for `i` but was sampled as another query's positive would be pushed away from `i` — a contradictory gradient. `τ = 0.05`: the gradient of `L_i` w.r.t. a negative's score is `softmax_j / τ`, so a small `τ` concentrates the gradient on the **hardest** negatives (the ones BM25 also confuses), which is exactly where the model can add something to BM25.

### Fusion with BM25 and model selection
Scores are z-normalised per query and fused, `s = α·z(dense) + (1−α)·z(BM25)`, `α` on a grid (and reciprocal-rank fusion, `Σ 1/(60 + rank)`, as a parameter-free check). Model selection per epoch and the choice of `α`, pooling and score mode are all by **dev hits@7** (equivalent to dev F1@7, since `F1@7 = 2H/(7N+G)`). All of these are chosen on the same 118 dev queries, so the dev numbers are slightly optimistic; the test submission is the honest number.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle, copy, hashlib, bisect, subprocess
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p7_dec02_lorma"
METHOD     = "lorma"          # Phase-7 list: LoRMA (compulsory). HP["lorma"]["variant"]: "plus" (LoRMA+) | "pi" (LoRMA_pi)
SMOKE_TEST = False            # True: SMOKE_STEPS steps, dev on SMOKE_DEV_Q queries (full dev pool), NO test / export

# ---- Phase-1 checkpoint (caps_decoder_only_architecture_02.ipynb: runs/arch02_hybrid_gqa/best.pt) -----------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-dec-02-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data + audit artifacts ----------------------------------------------------------------------
HF_REPO        = "Exploration-Lab/CS-781-Capstone"
LOCAL_DATA_DIR = None
P7    = "Phase_7_PCR"
FILES = {"train": ("Train/train.json", "Train/train_candidates.json"),
         "dev":   ("Dev/dev.json",     "Dev/dev_candidates.json"),
         "test":  ("Test/test.json",   "Test/test_candidates.json")}
BM25_PATH   = Path("/kaggle/input/datasets/ascyrax/caps-p7-bm25/bm25_top300.json") if IS_KAGGLE else Path("./input/bm25_top300.json")
TOKCACHE_IN = Path("/kaggle/input/datasets/ascyrax/caps-p7-tokcache") if IS_KAGGLE else Path("./input/tokcache")   # optional (written by run #1)
N_PROC      = 4

# ---- Views ----------------------------------------------------------------------------------------
CHUNK_TOKENS   = 512          # incl. [BOS] and [EOS]
MAX_DOC_CHUNKS = 16           # audit: 16 x 512 reads 86 % of all tokens
QUERY_VIEW     = "ctx+head"   # "ctx" (sentences around <CITATION>) | "head" (same as a candidate) | "ctx+head" (1 head chunk + ctx)
CTX_WINDOW     = 1            # sentences kept on each side of a <CITATION> sentence
Q_MAX_CHUNKS   = 4
CIT_TAG        = "<CITATION>"

# ---- Encoder / scoring ----------------------------------------------------------------------------
POOLING    = "auto"           # "mean" | "wmean" | "last" | "auto" (best zero-shot pooling on dev)
SCORE_MODE = "auto"           # "mean" | "maxsim" | "auto" (better fused dev hits@7 after training)
USE_HEAD   = True             # d x d linear head, identity at init
TAU        = 0.05
EMB_BATCH  = 64               # sequences per inference forward

# ---- Training -------------------------------------------------------------------------------------
EPOCHS        = 4
BATCH_PAIRS   = 16            # (query, positive) pairs per step; + 1 BM25 hard negative each -> 2*B documents
C_TRAIN       = 4             # random chunks per document per step (resampled every step)
QC_TRAIN      = 2             # random query-view chunks per query per step
HN_TOP        = 50            # hard negatives are drawn from the query's BM25 top-HN_TOP that are not gold
FWD_MICRO     = 32            # sequences per forward piece inside a step (lower on OOM; does not change the maths)
WARMUP_FRAC   = 0.05
GRAD_CLIP     = 1.0
GRAD_CKPT     = True          # activation checkpointing per block (needed for ~160 x 512 tokens per step on a T4)
TIME_BUDGET_MIN = 360
HP = {"lorma": dict(lr=5e-4, head_lr=2e-4, wd=0.0, r=16, alpha=32, variant="plus")}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")

# ---- Fusion / evaluation --------------------------------------------------------------------------
K_EVAL     = 7
ALPHA_GRID = tuple(np.round(np.arange(0.0, 1.0001, 0.05), 2))
RRF_K      = 60
RECALL_KS  = (7, 20, 50, 100, 300)

# ---- Exports --------------------------------------------------------------------------------------
EXPORT_TEACHER = True         # dense scores over every train query's BM25 top-300 (+ its gold) for the KD run
SAVE_SCORES    = True         # dense top-300 per dev/test query (for later ensembles)
SAVE_TOP       = 300

# ---- Smoke test -----------------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_DEV_Q = 40, 40

# ---- derived --------------------------------------------------------------------------------------
assert METHOD in HP and POOLING in ("mean", "wmean", "last", "auto") and SCORE_MODE in ("mean", "maxsim", "auto")
assert QUERY_VIEW in ("ctx", "head", "ctx+head")
BODY = CHUNK_TOKENS - 2
MAX_DOC_TOK = MAX_DOC_CHUNKS * BODY
MAX_DOC_CHARS = int(MAX_DOC_TOK * 4.6)        # audit: 4.35-4.44 chars/token; a little slack, the token cap is exact
RUN_DIR = WORK_DIR / "runs" / RUN_NAME; RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | {METHOD} {HP[METHOD]} | chunks {CHUNK_TOKENS} x <= {MAX_DOC_CHUNKS} | query view {QUERY_VIEW} | "
      f"B={BATCH_PAIRS} pairs, C={C_TRAIN}, QC={QC_TRAIN}, tau={TAU} | epochs {EPOCHS} | smoke={SMOKE_TEST}")

RUN p7_dec02_lorma | lorma {'lr': 0.0005, 'head_lr': 0.0002, 'wd': 0.0, 'r': 16, 'alpha': 32, 'variant': 'plus'} | chunks 512 x <= 16 | query view ctx+head | B=16 pairs, C=4, QC=2, tau=0.05 | epochs 4 | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- as in Phases 3-4.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_decoder_only_architecture_02.ipynb` (Cells 4-11)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# Cell 4 — Model config (serialisable; saved inside every checkpoint)
@dataclass
class ModelConfig:
    # --- shape ---
    vocab_size: int = 32000
    n_layer: int = 12
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8                 # < n_head  => GQA (each KV head is shared by n_head//n_kv_head query heads)
    head_dim: int = 64
    ff_mult: float = 8 / 3             # SwiGLU hidden size ~ (8/3)*d_model, rounded up to a multiple of 64
    d_ff: Optional[int] = None         # if set, overrides ff_mult
    max_seq_len: int = 1024

    # --- attention pattern ---
    local_window: int = 0              # 0 => every layer is full causal attention
    global_layers: List[int] = field(default_factory=list)   # 0-indexed layers that stay FULL when local_window > 0

    # --- components ---
    norm_type: str = "rmsnorm"         # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"             # "rope" | "none"
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"           # "swiglu" | "gelu"
    mixer_type: str = "attention"      # token-mixer registry key (swap in other mixers later)
    channel_mixer_type: str = "mlp"    # channel-mixer registry key

    # --- misc ---
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim, "d_model must equal n_head*head_dim"
        assert self.n_head % self.n_kv_head == 0, "n_head must be divisible by n_kv_head"
        assert self.pos_type in ("rope", "none")
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)
        if self.local_window > 0:
            assert all(0 <= i < self.n_layer for i in self.global_layers)

    def is_global(self, layer_idx: int) -> bool:
        return self.local_window <= 0 or layer_idx in self.global_layers

    def window_for(self, layer_idx: int) -> Optional[int]:
        return None if self.is_global(layer_idx) else self.local_window


def preset_baseline_rope(vocab_size=None, max_seq_len=1024):
    """Model 01's shape (MHA 8/8, d_ff 1408, all layers full) but with RoPE really applied.
    Use this to train a clean controlled baseline for the ablation if you have GPU time."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len)


def preset_hybrid_gqa(vocab_size=None, max_seq_len=1024):
    """Architecture 02: GQA (2 KV heads) + sliding window 256 on 9 of 12 layers + wider FFN to keep params equal."""
    return ModelConfig(vocab_size=vocab_size or VOCAB_SIZE, max_seq_len=max_seq_len,
                       n_kv_head=2, d_ff=1664, local_window=256, global_layers=[3, 7, 11])

print(json.dumps(asdict(preset_hybrid_gqa()), indent=1))

{
 "vocab_size": 32000,
 "n_layer": 12,
 "d_model": 512,
 "n_head": 8,
 "n_kv_head": 2,
 "head_dim": 64,
 "ff_mult": 2.6666666666666665,
 "d_ff": 1664,
 "max_seq_len": 1024,
 "local_window": 256,
 "global_layers": [
  3,
  7,
  11
 ],
 "norm_type": "rmsnorm",
 "norm_eps": 1e-05,
 "pos_type": "rope",
 "rope_theta": 10000.0,
 "mlp_type": "swiglu",
 "mixer_type": "attention",
 "channel_mixer_type": "mlp",
 "bias": false,
 "attn_dropout": 0.0,
 "resid_dropout": 0.0,
 "embd_dropout": 0.0,
 "tie_embeddings": true,
 "init_std": 0.02,
 "use_sdpa": true
}


In [6]:
# Cell 5 — Normalisation layers
class LayerNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(var + self.eps) * self.g + self.b


class RMSNorm(nn.Module):
    """RMSNorm: scale by 1/rms(x) only (no mean centering, no bias). Computed in fp32 for stability."""
    def __init__(self, d: int, eps: float = 1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)
        return (x / rms) * self.g


def build_norm(cfg: ModelConfig):
    if cfg.norm_type == "layernorm": return LayerNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "rmsnorm":   return RMSNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)

In [7]:
# Cell 6 — Rotary position embeddings (RoPE)
class RoPE(nn.Module):
    """Rotates each (even, odd) pair of a q/k vector by an angle proportional to its position.
    Then q_i . k_j depends only on the RELATIVE offset i-j. Tables are precomputed; math runs in fp32."""
    def __init__(self, head_dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)       # [max_seq_len, head_dim/2]
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):                      # x: [B,H,T,D]; cos,sin: [*,1,T,D/2]
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids):
        """q: [B,Hq,T,D], k: [B,Hkv,T,D], position_ids: [B,T] (absolute positions of these T tokens)."""
        cos = self.cos[position_ids].unsqueeze(1).float()   # [B,1,T,D/2]
        sin = self.sin[position_ids].unsqueeze(1).float()
        dt = q.dtype
        return (self._rotate(q.float(), cos, sin).to(dt),
                self._rotate(k.float(), cos, sin).to(dt))

In [8]:
# Cell 7 — Attention: GQA + optional sliding window + KV cache
_MASK_CACHE = {}

def causal_window_mask(T: int, S: int, window: Optional[int], dev) -> torch.Tensor:
    """Bool mask [1,1,T,S], True = query may attend to key.
    The T queries are the LAST T positions of the S keys (so it also works with a KV cache).
    Query at absolute position p attends keys j with  j <= p  and  (window is None or p - j < window),
    i.e. itself + the previous (window-1) tokens."""
    key = (T, S, window, str(dev))
    m = _MASK_CACHE.get(key)
    if m is None:
        qpos = torch.arange(S - T, S, device=dev).view(T, 1)
        kpos = torch.arange(S, device=dev).view(1, S)
        allowed = kpos <= qpos
        if window is not None:
            allowed = allowed & ((qpos - kpos) < window)
        m = allowed.view(1, 1, T, S)
        if T > 1:                       # cache the big square-ish masks only
            _MASK_CACHE[key] = m
    return m


class Attention(nn.Module):
    """Causal multi-head self-attention.
    - GQA: n_kv_head K/V heads, each shared by n_head // n_kv_head query heads (KV cache shrinks by that factor).
    - Local layers: each token sees only the last `window` tokens. Global layers: window=None (full causal).
    - KV cache stores the (already RoPE-rotated) K and V."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.cfg = cfg
        self.layer_idx = layer_idx
        self.window = cfg.window_for(layer_idx)
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.attn_dropout = nn.Dropout(cfg.attn_dropout)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, n):
        B, T, _ = x.shape
        return x.view(B, T, n, self.head_dim).transpose(1, 2)            # [B,n,T,D]

    def forward(self, x, rope=None, position_ids=None, attention_mask=None, kv_cache=None):
        """x: [B,T,d]; position_ids: [B,T]; attention_mask: [B,S] (1 keep / 0 pad, S = cache + T);
        kv_cache: (k_prev, v_prev) each [B,n_kv_head,S_prev,D]. Returns (out, new_cache)."""
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        k = self._split(self.k_proj(x), self.n_kv_head)
        v = self._split(self.v_proj(x), self.n_kv_head)

        if rope is not None:                                              # RoPE on the NEW q/k only
            q, k = rope(q, k, position_ids)

        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        S = k.shape[2]

        if self.n_kv_head != self.n_head:                                 # GQA: share each K/V head
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        window_binds = self.window is not None and S > self.window
        need_mask = (attention_mask is not None) or window_binds or (T > 1 and S != T)

        allowed = None
        if need_mask:
            allowed = causal_window_mask(T, S, self.window, x.device)     # [1,1,T,S]
            if attention_mask is not None:
                allowed = allowed & attention_mask[:, None, None, :].bool()
                # a left-padding query would see nothing (-> NaN). Let every query see itself (harmless for real tokens).
                qpos = torch.arange(S - T, S, device=x.device).view(T, 1)
                kpos = torch.arange(S, device=x.device).view(1, S)
                allowed = allowed | (kpos == qpos).view(1, 1, T, S)

        p = self.attn_dropout.p if self.training else 0.0
        if self.cfg.use_sdpa:
            if need_mask:
                out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, dropout_p=p, is_causal=False)
            else:                                                         # plain causal (global layer, training)
                out = F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=(T > 1))
        else:                                                             # manual reference path (used in tests)
            if allowed is None:
                allowed = causal_window_mask(T, S, None, x.device)
            scores = (q @ k.transpose(-2, -1)) * self.scale
            scores = scores.masked_fill(~allowed, float("-inf"))
            probs = self.attn_dropout(F.softmax(scores.float(), dim=-1).to(q.dtype))
            out = probs @ v

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache

In [9]:
# Cell 8 — Feed-forward blocks
class SwiGLU(nn.Module):
    """down( silu(gate(x)) * up(x) )"""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GELUMLP(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.fc2 = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.fc2(F.gelu(self.fc1(x))))

In [10]:
# Cell 9 — Registries (plug in new token mixers / channel mixers later) + transformer block
TOKEN_MIXERS, CHANNEL_MIXERS = {}, {}

def register_token_mixer(name):
    def deco(fn): TOKEN_MIXERS[name] = fn; return fn
    return deco

def register_channel_mixer(name):
    def deco(fn): CHANNEL_MIXERS[name] = fn; return fn
    return deco

@register_token_mixer("attention")
def _build_attention(cfg, layer_idx):
    return Attention(cfg, layer_idx)

@register_channel_mixer("mlp")
def _build_mlp(cfg):
    if cfg.mlp_type == "swiglu": return SwiGLU(cfg)
    if cfg.mlp_type == "gelu":   return GELUMLP(cfg)
    raise ValueError(cfg.mlp_type)


class Block(nn.Module):
    """Pre-norm block:  x = x + mixer(norm(x));  x = x + mlp(norm(x))."""
    def __init__(self, cfg: ModelConfig, layer_idx: int):
        super().__init__()
        self.norm1, self.norm2 = build_norm(cfg), build_norm(cfg)
        self.mixer = TOKEN_MIXERS[cfg.mixer_type](cfg, layer_idx)
        self.cmix = CHANNEL_MIXERS[cfg.channel_mixer_type](cfg)
    def forward(self, x, **kw):
        h, new_cache = self.mixer(self.norm1(x), **kw)
        x = x + h
        x = x + self.cmix(self.norm2(x))
        return x, new_cache

In [11]:
# Cell 10 — Backbone, LM head, weight init
class Backbone(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.n_layer)])
        self.norm_f = build_norm(cfg)

    def forward(self, input_ids, attention_mask=None, position_ids=None, kv_caches=None):
        B, T = input_ids.shape
        if position_ids is None:                       # default: continue after whatever is in the cache
            past = kv_caches[0][0].shape[2] if kv_caches is not None else 0
            position_ids = (past + torch.arange(T, device=input_ids.device)).unsqueeze(0).expand(B, T)
        x = self.emb_drop(self.tok_emb(input_ids))
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, rope=self.rope, position_ids=position_ids, attention_mask=attention_mask,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class LegalLM(nn.Module):
    """Backbone + LM head tied to the token embedding."""
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.backbone = Backbone(cfg)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.backbone.tok_emb.weight

    def forward(self, input_ids, targets=None, attention_mask=None, position_ids=None, kv_caches=None):
        h, new_caches = self.backbone(input_ids, attention_mask=attention_mask,
                                      position_ids=position_ids, kv_caches=kv_caches)
        logits = self.lm_head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss, new_caches


def init_weights(model: LegalLM, cfg: ModelConfig):
    """N(0, init_std) everywhere; residual-branch output projections scaled by 1/sqrt(2*n_layer)."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    scale = 1.0 / math.sqrt(2 * cfg.n_layer)
    with torch.no_grad():
        for blk in model.backbone.blocks:
            if hasattr(blk.mixer, "o_proj"):  blk.mixer.o_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "down_proj"): blk.cmix.down_proj.weight.mul_(scale)
            if hasattr(blk.cmix, "fc2"):       blk.cmix.fc2.weight.mul_(scale)

In [12]:
# Cell 11 — Utilities: parameter count, optimizer groups (FIXED), save / load
def count_params(model: nn.Module) -> Dict[str, int]:
    """Tied embedding counted once (model.parameters() de-duplicates shared tensors)."""
    total = sum(p.numel() for p in model.parameters())
    emb = model.backbone.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


def build_param_groups(model: nn.Module, weight_decay: float):
    """Weight decay on 2-D weight matrices only (attention/MLP projections).
    No decay on norm gains/biases (1-D) and on the embedding.
    (Model 01 matched '.b' in names, which also matched 'backbone.blocks...' -> decay was never applied.)"""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "tok_emb" not in name:
            decay.append(p)
        else:
            no_decay.append(p)
    return [{"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0}]


def save_model(path, model, cfg, extra=None):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "extra": extra or {}}, tmp)
    os.replace(tmp, path)


def load_model(path, map_location="cpu"):
    ckpt = torch.load(path, map_location=map_location)
    cfg = ModelConfig(**ckpt["config"])
    model = LegalLM(cfg)
    model.load_state_dict(ckpt["model"])
    return model, cfg, ckpt.get("extra", {})

## Load the Phase-1 checkpoint

In [13]:
# Cell 12 — Load the Phase-1 checkpoint and sanity-check it
_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
lm_cfg = ModelConfig(**_ck["config"])
lm = LegalLM(lm_cfg)
lm.load_state_dict(_ck["model"])
lm_extra = _ck.get("extra", {})
del _ck
lm = lm.to(device).eval()
pc = count_params(lm)
print(f"Phase-1 model | pos_type={lm_cfg.pos_type} | layers={lm_cfg.n_layer} d_model={lm_cfg.d_model} | "
      f"heads {lm_cfg.n_head}:{lm_cfg.n_kv_head} | d_ff={lm_cfg.d_ff} | local_window={lm_cfg.local_window} "
      f"global_layers={lm_cfg.global_layers} | max_seq_len {lm_cfg.max_seq_len} | params {pc['total']/1e6:.2f}M")
print("checkpoint extra: step", lm_extra.get("step"), "| val_loss", lm_extra.get("val_loss"))
_sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
_ids = [BOS_ID] + tokenizer.encode(_sent)
_x = torch.tensor([_ids[:-1]], device=device); _y = torch.tensor([_ids[1:]], device=device)
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
    _, _l, _ = lm(_x, targets=_y)
print(f"LM loss on a legal sentence: {_l.item():.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
assert _l.item() < 9.0, "checkpoint looks untrained / mis-loaded"
assert VOCAB_SIZE < 65536, "token ids are stored as uint16"

Phase-1 model | pos_type=rope | layers=12 d_model=512 | heads 8:2 | d_ff=1664 | local_window=256 global_layers=[3, 7, 11] | max_seq_len 1024 | params 54.93M
checkpoint extra: step 12749 | val_loss 1.9441224113106728
LM loss on a legal sentence: 1.888  (ln V = 10.37)


## Phase-7 data (the audit's loader) and the BM25 lists
The loader is the audit's (Cell 4), unchanged. IDs are matched on a canonical key (leading zeros stripped); every output uses the pool's own ID string. The BM25 lists come from the audit's Cell 12 (`bm25_top300.json`, best configuration, no year filter).

In [14]:
# Cell 13 — Load the 6 files (audit loader) + BM25 lists
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

def fetch(rel):
    rel = f"{P7}/{rel}"
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet, or download {P7} and set LOCAL_DATA_DIR.") from ex

def read_json_any(path):
    txt = Path(path).read_text(encoding="utf-8")
    try: return json.loads(txt)
    except json.JSONDecodeError: return [json.loads(l) for l in txt.splitlines() if l.strip()]

ID_KEYS   = ("id", "query_id", "case_id", "candidate_id", "doc_id", "document_id", "qid", "cid")
TEXT_KEYS = ("text", "case_text", "document", "content", "sentences", "judgment", "judgement", "body")
GOLD_KEYS = ("relevant_candidates", "relevant", "relevant_ids", "positives", "labels", "gold", "citations", "cited")
def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default
def as_sents(v):
    if isinstance(v, str): return [s for s in v.split("\n")]
    out = []
    for x in v:
        if isinstance(x, list): out.extend(str(y) for y in x)
        elif isinstance(x, dict): out.append(str(_first(x, TEXT_KEYS + ("sentence",), "")))
        else: out.append(str(x))
    return out
def canon(x): s = str(x).strip(); return s.lstrip("0") or "0"
def unwrap(raw):
    if isinstance(raw, dict) and len(raw) == 1:
        v = next(iter(raw.values()))
        if isinstance(v, (list, dict)) and len(v) > 1: return v
    if isinstance(raw, dict) and isinstance(raw.get("data"), list): return raw["data"]
    return raw
def items_of(raw):
    raw = unwrap(raw)
    if isinstance(raw, dict):
        out = []
        for k, v in raw.items():
            if isinstance(v, dict): v = dict(v); v.setdefault("id", k)
            else: v = {"id": k, "text": v}
            out.append(v)
        return out
    return list(raw)

DOCS = {}                      # canonical id -> {"id": original string, "sents": [...]}, shared by all splits (pools overlap)
POOL, QRY = {}, {}             # split -> list of canonical ids (pool order) ; split -> list of query dicts
for split, (qf, cf) in FILES.items():
    raw_c = read_json_any(fetch(cf)); order, seen = [], set()
    for e in items_of(raw_c):
        k = canon(_first(e, ID_KEYS))
        if k in seen: continue
        seen.add(k); order.append(k)
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS)), sents=as_sents(_first(e, TEXT_KEYS)))
    POOL[split] = order; del raw_c
    raw_q = read_json_any(fetch(qf)); qs = []
    pool_set = set(order)
    for i, e in enumerate(items_of(raw_q)):
        k = canon(_first(e, ID_KEYS, i))
        if k not in DOCS: DOCS[k] = dict(id=str(_first(e, ID_KEYS, i)), sents=as_sents(_first(e, TEXT_KEYS)))
        g = _first(e, GOLD_KEYS)
        gold = None
        if split != "test":
            g = [g] if isinstance(g, (str, int)) else (g or [])
            gold = []
            for x in g:
                c = canon(x)
                if c in pool_set and c not in gold: gold.append(c)
        qs.append(dict(key=k, id=str(_first(e, ID_KEYS, i)), gold=gold))
    QRY[split] = qs; del raw_q; gc.collect()
    print(f"{split:5s}: {len(qs)} queries | pool {len(order)} | queries inside their own pool: {sum(q['key'] in pool_set for q in qs)}"
          + (f" | gold pairs {sum(len(q['gold']) for q in qs)} | queries without gold {sum(not q['gold'] for q in qs)}" if split != "test" else ""))
print("distinct documents over all splits:", len(DOCS))

_bm = json.loads(BM25_PATH.read_text())
BM25_CFG, BM25 = _bm["config"], {s: {canon(q): [(canon(c), float(v)) for c, v in lst] for q, lst in d.items()} for s, d in _bm["lists"].items()}
del _bm
print("BM25 config:", BM25_CFG)
for s in FILES:
    miss = sum(q["key"] not in BM25[s] for q in QRY[s]); assert miss == 0, f"{s}: {miss} queries have no BM25 list"

Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

train: 827 queries | pool 4320 | queries inside their own pool: 827 | gold pairs 5308 | queries without gold 10


Phase_7_PCR/Dev/dev_candidates.json: reconstructing file:   0%|          |  0.00B / 51.9MB            

... (3 outputs left out)

Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

test : 237 queries | pool 1727 | queries inside their own pool: 237
distinct documents over all splits: 5825
BM25 config: {'ng': 'bi', 'k1': 8.0, 'b': 1.0, 'mode': 'binary', 'H@7': 395, 'F1@7': 0.43913285158421345, 'recall@100': 0.7543679342240494, 'MRR': 0.8950390337803802, 'year_slack': None}


## Tokenisation, views, cache
Every distinct document is tokenised once (pools overlap) into its **document view** (head 2/3 + tail 1/3 of `MAX_DOC_CHARS` characters, then at most `MAX_DOC_CHUNKS × 510` tokens), and every query additionally into its **evidence view** (the `<CITATION>` sentences ± `CTX_WINDOW`, in document order, at most `Q_MAX_CHUNKS × 510` tokens). The cache is keyed by the view settings and written to `WORK_DIR/tokcache/`. **Upload that folder as the Kaggle dataset `caps-p7-tokcache`**: every later Phase-7 run uses the same tokenizer and reads it instead of re-tokenising (~5–15 min saved per run).

In [15]:
# Cell 14 — Tokenise (multiprocessing) + cache; chunking
def cut_text(t):
    if len(t) <= MAX_DOC_CHARS: return t
    h = 2 * MAX_DOC_CHARS // 3
    return t[:h] + "\n" + t[-(MAX_DOC_CHARS - h):]
def ctx_text(sents):
    hit = [j for j, s in enumerate(sents) if CIT_TAG in s]
    keep = sorted({k for j in hit for k in range(max(0, j - CTX_WINDOW), min(len(sents), j + CTX_WINDOW + 1))})
    return "\n".join(sents[k] for k in keep)

_W_ITEMS = None
def _tok_work(i):
    k, want_ctx = _W_ITEMS[i]; sents = DOCS[k]["sents"]
    doc = np.asarray(tokenizer.encode(cut_text("\n".join(sents)))[:MAX_DOC_TOK], dtype=np.uint16)
    ctx = None
    if want_ctx:
        t = ctx_text(sents)
        ctx = np.asarray(tokenizer.encode(t[: int(Q_MAX_CHUNKS * BODY * 4.6)])[: Q_MAX_CHUNKS * BODY], dtype=np.uint16) if t else np.zeros(0, np.uint16)
    return k, doc, ctx

SIG = hashlib.md5(json.dumps([CHUNK_TOKENS, MAX_DOC_CHUNKS, CTX_WINDOW, Q_MAX_CHUNKS, CIT_TAG, VOCAB_SIZE]).encode()).hexdigest()[:10]
CACHE_NAME = f"p7_tok_{SIG}.pkl"
cache_out = WORK_DIR / "tokcache" / CACHE_NAME; cache_out.parent.mkdir(parents=True, exist_ok=True)
TOKD, TOKQ = {}, {}
for p in (TOKCACHE_IN / CACHE_NAME, cache_out):
    if p.exists():
        TOKD, TOKQ = pickle.loads(p.read_bytes()); print(f"token cache loaded: {p} ({len(TOKD)} docs, {len(TOKQ)} query views)"); break
qkeys = {q["key"] for s in FILES for q in QRY[s]}
todo = [(k, k in qkeys) for k in DOCS if k not in TOKD or (k in qkeys and k not in TOKQ)]
if todo:
    t0 = time.time(); _W_ITEMS = todo
    with mp.get_context("fork").Pool(N_PROC) as pool:
        for k, d, c in tqdm(pool.imap_unordered(_tok_work, range(len(todo)), chunksize=4), total=len(todo), desc="tokenise"):
            TOKD[k] = d
            if c is not None: TOKQ[k] = c
    cache_out.write_bytes(pickle.dumps((TOKD, TOKQ), protocol=4))
    print(f"tokenised {len(todo)} documents in {time.time() - t0:.0f}s -> {cache_out} ({cache_out.stat().st_size / 1e6:.0f} MB)")

def chunks_of(ids, max_chunks):
    ids = ids[: max_chunks * BODY]
    ch = [ids[i:i + BODY] for i in range(0, len(ids), BODY)]
    if len(ch) > 1 and len(ch[-1]) < 32: ch = ch[:-1]          # a 5-token tail chunk is noise
    return ch or [np.zeros(0, np.uint16)]
def doc_chunks(k): return chunks_of(TOKD[k], MAX_DOC_CHUNKS)
def query_chunks(k):
    head = chunks_of(TOKD[k], 1)
    ctx = chunks_of(TOKQ[k], Q_MAX_CHUNKS) if len(TOKQ.get(k, ())) else []
    if QUERY_VIEW == "head": return chunks_of(TOKD[k], Q_MAX_CHUNKS)
    if QUERY_VIEW == "ctx": return ctx or head
    return (head + ctx)[:Q_MAX_CHUNKS]
_nq = [len(query_chunks(q["key"])) for q in QRY["train"]]; _nd = [len(doc_chunks(k)) for k in POOL["train"]]
print(f"train: query chunks/query mean {np.mean(_nq):.2f} | doc chunks/doc mean {np.mean(_nd):.2f} (cap {MAX_DOC_CHUNKS}) | "
      f"queries with an empty ctx view: {sum(len(TOKQ.get(q['key'], ())) == 0 for q in QRY['train'])}")

tokenise:   0%|          | 0/5825 [00:00<?, ?it/s]

tokenised 5825 documents in 16s -> /kaggle/working/tokcache/p7_tok_18962ef646.pkl (61 MB)
train: query chunks/query mean 3.70 | doc chunks/doc mean 9.98 (cap 16) | queries with an empty ctx view: 1


## The adapted model, embedding, scoring, evaluation, fusion

In [16]:
# Cell 15 — LoRMA (the Phase-3/4/5/6 code, unchanged) + bi-encoder construction
_PI_IDX = {}
def _pi_index(d, dev):
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    # h = I(B A) (W0 x).  'plus': y + (alpha/r) B (A y), B = 0 at init.  'pi': rotated rows of BA, identity at init.
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))
        return y @ Tm.t()

def apply_adapter(module, targets, make):
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n


def build_model():
    # Fresh copy of the Phase-1 weights; everything frozen except the LoRMA factors and the (identity-initialised) head.
    model = copy.deepcopy(lm).cpu()
    for p in model.parameters(): p.requires_grad = False
    hp = HP[METHOD]
    ref = copy.deepcopy(model.backbone)
    n = apply_adapter(model.backbone, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
    print(f"LoRMA[{hp['variant']}] on {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
    _ids = torch.randint(5, VOCAB_SIZE, (1, 48))
    with torch.no_grad():
        a, _ = model.backbone.eval()(_ids); b, _ = ref.eval()(_ids)
    err = (a - b).abs().max().item()
    print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
    assert err < 1e-4, "adapter does not start from the Phase-1 function"
    del ref
    d = lm_cfg.d_model
    model.head = nn.Linear(d, d, bias=False) if USE_HEAD else None
    if USE_HEAD:
        with torch.no_grad(): model.head.weight.copy_(torch.eye(d))
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.3f}M ({100*tr/tot:.2f}%) | head {d*d/1e6 if USE_HEAD else 0:.3f}M")
    model.n_total, model.n_trainable = tot, tr
    return model

def backbone_hidden(model, ids, mask=None, pos=None):
    # Backbone.forward without KV cache, with optional activation checkpointing (training only).
    bb = model.backbone; N, T = ids.shape
    if pos is None: pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(N, T)
    x = bb.emb_drop(bb.tok_emb(ids))
    for blk in bb.blocks:
        if model.training and GRAD_CKPT and torch.is_grad_enabled():
            x = checkpoint(lambda t, blk=blk: blk(t, rope=bb.rope, position_ids=pos, attention_mask=mask)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, rope=bb.rope, position_ids=pos, attention_mask=mask)
    return bb.norm_f(x)

In [17]:
# Cell 16 — Pooling, chunk embedding, document scoring (mean / maxsim), evaluation, fusion
def make_batch(seqs):
    # seqs: list of uint16 arrays (chunk bodies). Returns ids [N, T] right-padded with [BOS] body [EOS], lengths [N].
    lens = [len(s) + 2 for s in seqs]; T = max(lens)
    ids = torch.full((len(seqs), T), PAD_ID, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, 0] = BOS_ID; ids[i, 1:1 + len(s)] = torch.from_numpy(s.astype(np.int64)); ids[i, 1 + len(s)] = EOS_ID
    return ids.to(device, non_blocking=True), torch.tensor(lens, device=device)

def pool_hidden(h, lens, how):
    # h [N, T, d] (right padding), lens [N] -> [N, d]
    N, T, _ = h.shape
    t = torch.arange(T, device=h.device).unsqueeze(0)
    m = (t < lens.unsqueeze(1)).to(h.dtype)
    if how == "mean":  w = m
    elif how == "wmean": w = m * (t + 1).to(h.dtype)                 # SGPT position weights
    elif how == "last":  return h[torch.arange(N, device=h.device), lens - 1]
    else: raise ValueError(how)
    return (h * w.unsqueeze(-1)).sum(1) / w.sum(1, keepdim=True)

def encode(model, ids, lens, poolings):
    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        h = backbone_hidden(model, ids)
    out = {}
    for p in poolings:
        e = pool_hidden(h.float(), lens, p)
        if getattr(model, "head", None) is not None: e = model.head(e)
        out[p] = F.normalize(e, dim=-1)
    return out

@torch.no_grad()
def embed_keys(model, keys, chunk_fn, poolings):
    # -> {pooling: [n_chunks, d] float32 (GPU)}, owner [n_chunks] (index into keys), sorted by owner
    model.eval()
    seqs, owner = [], []
    for i, k in enumerate(keys):
        for c in chunk_fn(k): seqs.append(c); owner.append(i)
    order = np.argsort([len(s) for s in seqs], kind="stable")
    E = {p: torch.empty(len(seqs), lm_cfg.d_model, device=device) for p in poolings}
    for b in range(0, len(order), EMB_BATCH):
        idx = order[b:b + EMB_BATCH]
        ids, lens = make_batch([seqs[j] for j in idx])
        out = encode(model, ids, lens, poolings)
        for p in poolings: E[p][torch.as_tensor(idx, device=device)] = out[p]
    return E, torch.tensor(owner, device=device)

def embed_split(model, split, poolings, q_subset=None):
    qs = QRY[split] if q_subset is None else QRY[split][:q_subset]
    t0 = time.time()
    Eq, oq = embed_keys(model, [q["key"] for q in qs], query_chunks, poolings)
    Ed, od = embed_keys(model, POOL[split], doc_chunks, poolings)
    return dict(Eq=Eq, oq=oq, Ed=Ed, od=od, nq=len(qs), nd=len(POOL[split]), secs=time.time() - t0, n_chunks=len(od) + len(oq))

def seg_mean(E, owner, n):
    s = torch.zeros(n, E.shape[1], device=E.device).index_add_(0, owner, E)
    return F.normalize(s, dim=-1)

@torch.no_grad()
def dense_scores(emb, pooling, mode, center=False, block=256):
    Eq, Ed, oq, od = emb["Eq"][pooling], emb["Ed"][pooling], emb["oq"], emb["od"]
    if center:
        mu = Ed.mean(0, keepdim=True); Eq = F.normalize(Eq - mu, dim=-1); Ed = F.normalize(Ed - mu, dim=-1)
    nq, nd = emb["nq"], emb["nd"]
    if mode == "mean":
        S = seg_mean(Eq, oq, nq) @ seg_mean(Ed, od, nd).T
    else:                                               # maxsim: mean over query chunks of the max over candidate chunks
        S = torch.zeros(nq, nd, device=device); cnt = torch.zeros(nq, device=device)
        for b in range(0, len(oq), block):
            sim = Eq[b:b + block] @ Ed.T                                        # [qb, n_doc_chunks]
            mx = torch.full((sim.shape[0], nd), -2.0, device=device).scatter_reduce(1, od.expand(sim.shape[0], -1), sim, "amax")
            S.index_add_(0, oq[b:b + block], mx); cnt.index_add_(0, oq[b:b + block], torch.ones(len(mx), device=device))
        S = S / cnt.clamp(min=1).unsqueeze(1)
    return S.float().cpu().numpy()

def self_mask(S, split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; S = S.copy()
    for i, q in enumerate(QRY[split][: S.shape[0]]):
        if q["key"] in col and not (q["gold"] and q["key"] in q["gold"]): S[i, col[q["key"]]] = -np.inf
    return S

def bm25_matrix(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    S = np.zeros((len(qs), len(POOL[split])), dtype=np.float32)
    for i, q in enumerate(qs):
        lst = BM25[split][q["key"]]; lo = min(v for _, v in lst) if lst else 0.0
        S[i, :] = lo - 1.0
        for c, v in lst:
            if c in col: S[i, col[c]] = v
    return S

def topk(S, k):
    k = min(k, S.shape[1]); idx = np.argpartition(-S, k - 1, axis=1)[:, :k]
    o = np.argsort(-np.take_along_axis(S, idx, 1), axis=1); return np.take_along_axis(idx, o, 1)
def gold_cols(split, nq=None):
    col = {k: j for j, k in enumerate(POOL[split])}; qs = QRY[split][:nq] if nq else QRY[split]
    return [set(col[g] for g in q["gold"]) for q in qs]
def evaluate(S, split, ks=(K_EVAL,), recall_ks=RECALL_KS):
    gc_ = gold_cols(split, S.shape[0]); N = len(gc_); G = sum(len(g) for g in gc_)
    T = topk(S, max(max(ks), max(recall_ks))); out = {}
    for k in ks:
        H = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N))
        out[f"H@{k}"] = H; out[f"P@{k}"] = H / (k * N); out[f"R@{k}"] = H / max(G, 1); out[f"F1@{k}"] = 2 * H / (k * N + G)
    for k in recall_ks: out[f"recall@{k}"] = sum(len(set(T[i, :k].tolist()) & gc_[i]) for i in range(N)) / max(G, 1)
    rr = [next((1 / (j + 1) for j, c in enumerate(T[i].tolist()) if c in gc_[i]), 0.0) for i in range(N)]
    out["MRR"] = float(np.mean(rr)); return out

def znorm(S):
    S = S.astype(np.float64); f = np.isfinite(S)
    m = np.where(f, S, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    v = np.where(f, (S - m) ** 2, 0).sum(1, keepdims=True) / f.sum(1, keepdims=True)
    return np.where(f, (S - m) / np.sqrt(v + 1e-12), -np.inf)
def rrf(*Ss):
    out = np.zeros_like(Ss[0], dtype=np.float64)
    for S in Ss:
        r = np.empty_like(S, dtype=np.int64); o = np.argsort(-S, axis=1); r[np.arange(len(S))[:, None], o] = np.arange(S.shape[1])
        out += 1.0 / (RRF_K + 1 + r)
    return out
def fuse_search(Sd, Sb, split):
    # Sd, Sb already self-masked. Returns best (alpha, eval) by H@7 (ties: recall@100), and the whole curve.
    zd, zb = znorm(Sd), znorm(Sb); curve = []
    for a in ALPHA_GRID:
        e = evaluate(a * np.nan_to_num(zd, neginf=-1e9) + (1 - a) * np.nan_to_num(zb, neginf=-1e9), split)
        curve.append((float(a), e["H@7"], e["F1@7"], e["recall@100"]))
    best = max(curve, key=lambda t: (t[1], t[3], -abs(t[0] - 0.5)))
    return best, curve
def fused(Sd, Sb, alpha):
    zd, zb = znorm(Sd), znorm(Sb)
    return alpha * np.nan_to_num(zd, neginf=-1e9) + (1 - alpha) * np.nan_to_num(zb, neginf=-1e9)

## Zero-shot transfer of the Phase-1 representations (before any training)
One pass over dev produces all three poolings (they share the hidden states). For each pooling and score mode we report **dense-only** dev metrics, the **best fusion with BM25**, and the effect of **centering** (subtracting the pool's mean chunk vector before the cosine). Centering targets *anisotropy*: language-model hidden states share a large common direction, so all cosines are close to each other and dominated by it (Ethayarajh 2019); removing the mean exposes the directions that actually differ between documents. If centering helps a lot zero-shot, that is direct evidence of anisotropy in the Phase-1 model.

In [18]:
# Cell 17 — Build the model; BM25-only dev reference; zero-shot table; choose POOLING
seed_everything(SEED)
model = build_model()
DEV_Q = SMOKE_DEV_Q if SMOKE_TEST else None
Sb_dev = self_mask(bm25_matrix("dev", DEV_Q), "dev")
e_b = evaluate(Sb_dev, "dev")
print(f"BM25 only (dev{'[:%d]' % DEV_Q if DEV_Q else ''}): H@7 {e_b['H@7']} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")

POOLS_ALL = ("mean", "wmean", "last")
emb0 = embed_split(model, "dev", POOLS_ALL, DEV_Q)
print(f"dev embedded: {emb0['n_chunks']} chunks in {emb0['secs']:.0f}s")
ZS = []
for p in POOLS_ALL:
    for mode in ("mean", "maxsim"):
        for center in (False, True):
            Sd = self_mask(dense_scores(emb0, p, mode, center), "dev")
            e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
            ZS.append(dict(pooling=p, mode=mode, center=center, H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], MRR=e["MRR"],
                           fused_H7=H, fused_F1=f1, alpha=a))
            print(f"  zero-shot {p:5s} {mode:6s} center={center!s:5s}: dense H@7 {e['H@7']:4d} F1 {e['F1@7']:.4f} r@100 {e['recall@100']:.3f} | "
                  f"fused H@7 {H:4d} F1 {f1:.4f} (alpha {a})")
if POOLING == "auto":
    POOLING = max(POOLS_ALL, key=lambda p: max(r["H7"] for r in ZS if r["pooling"] == p))
print("POOLING ->", POOLING)
del emb0; gc.collect(); torch.cuda.empty_cache() if device.type == "cuda" else None

LoRMA[plus] on 84 Linear layers (r=16, alpha=32)
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 57.16M | trainable 2.228M (3.90%) | head 0.262M
BM25 only (dev): H@7 395 F1@7 0.4391 recall@100 0.754 MRR 0.895
dev embedded: 11164 chunks in 76s
  zero-shot mean  mean   center=False: dense H@7  216 F1 0.2401 r@100 0.632 | fused H@7  397 F1 0.4414 (alpha 0.3)
  zero-shot mean  mean   center=True : dense H@7  221 F1 0.2457 r@100 0.637 | fused H@7  397 F1 0.4414 (alpha 0.1)
  zero-shot mean  maxsim center=False: dense H@7  218 F1 0.2424 r@100 0.616 | fused H@7  398 F1 0.4425 (alpha 0.3)
  zero-shot mean  maxsim center=True : dense H@7  223 F1 0.2479 r@100 0.630 | fused H@7  398 F1 0.4425 (alpha 0.15)
  zero-shot wmean mean   center=False: dense H@7  218 F1 0.2424 r@100 0.643 | fused H@7  397 F1 0.4414 (alpha 0.35)
  zero-shot wmean mean   center=True : dense H@7  227 F1 0.2524 r@100 0.654 | fused H@7  397 F1 0.4414 (alpha 0.05)
  zero-shot wmean maxsim center=False: d

## Training
Each step: `BATCH_PAIRS` (query, positive) pairs from a shuffled list of all train gold pairs (an epoch visits every gold pair once, so queries with many citations get proportionally more steps — consistent with the micro-averaged metric). For each pair one hard negative is drawn from the query's BM25 top-`HN_TOP` minus its gold and itself. Every document contributes `C_TRAIN` random chunks and every query `QC_TRAIN` random evidence-view chunks (fresh random choices every step = augmentation). Embeddings are the normalised means of the chunk vectors. The forward runs in pieces of `FWD_MICRO` sequences with activation checkpointing; the loss needs all `2B` documents together, so the backward is one call.

In [19]:
# Cell 18 — Training loop (multi-positive InfoNCE with false-negative masking), per-epoch dev evaluation
TRQ = [q for q in QRY["train"] if q["gold"]]
TR_PAIRS = [(qi, g) for qi, q in enumerate(TRQ) for g in q["gold"]]
TR_GOLD = [set(q["gold"]) for q in TRQ]
TR_HN = [[c for c, _ in BM25["train"][q["key"]][:HN_TOP] if c not in TR_GOLD[qi] and c != q["key"]] for qi, q in enumerate(TRQ)]
print(f"train: {len(TRQ)} queries with gold | {len(TR_PAIRS)} (query, positive) pairs | queries without a hard negative: {sum(not h for h in TR_HN)}")

def sample_chunks(ch, n, rng): return ch if len(ch) <= n else [ch[i] for i in sorted(rng.sample(range(len(ch)), n))]

def step_loss(model, batch, rng):
    # batch: list of (qi, pos_key). Returns loss, stats.
    docs = []
    for qi, pk in batch:
        docs.append(pk)
        hn = TR_HN[qi]; docs.append(rng.choice(hn) if hn else rng.choice(POOL["train"]))
    seqs, owner = [], []
    for i, (qi, _) in enumerate(batch):
        for c in sample_chunks(query_chunks(TRQ[qi]["key"]), QC_TRAIN, rng): seqs.append(c); owner.append(i)
    nQ = len(batch)
    for j, k in enumerate(docs):
        for c in sample_chunks(doc_chunks(k), C_TRAIN, rng): seqs.append(c); owner.append(nQ + j)
    embs = []
    for b in range(0, len(seqs), FWD_MICRO):
        ids, lens = make_batch(seqs[b:b + FWD_MICRO]); embs.append(encode(model, ids, lens, (POOLING,))[POOLING])
    E = torch.cat(embs); own = torch.tensor(owner, device=device)
    V = seg_mean(E, own, nQ + len(docs)); Q, D = V[:nQ], V[nQ:]
    logits = (Q @ D.T) / TAU                                                  # [B, 2B]
    mask = torch.zeros_like(logits, dtype=torch.bool)
    for i, (qi, _) in enumerate(batch):
        for j, k in enumerate(docs):
            if j != 2 * i and (k in TR_GOLD[qi] or k == TRQ[qi]["key"]): mask[i, j] = True
    logits = logits.masked_fill(mask, float("-inf"))
    target = torch.arange(nQ, device=device) * 2
    loss = F.cross_entropy(logits.float(), target)
    with torch.no_grad():
        acc = (logits.argmax(1) == target).float().mean().item()
        gap = (logits[torch.arange(nQ), target] - logits[torch.arange(nQ), target + 1]).mean().item() * TAU   # cos(q,pos) - cos(q,hard neg)
    return loss, dict(acc=acc, gap=gap, masked=int(mask.sum().item()), seqs=len(seqs))

def param_groups(model):
    hp = HP[METHOD]
    head = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("head.")]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("head.")]
    g = [dict(params=rest, lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"])]
    if head: g.append(dict(params=head, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0))
    return g

def dev_eval(model, emb=None):
    emb = emb or embed_split(model, "dev", (POOLING,), DEV_Q)
    out = {}
    for mode in ("mean", "maxsim"):
        Sd = self_mask(dense_scores(emb, POOLING, mode), "dev")
        e = evaluate(Sd, "dev"); (a, H, f1, r100), _ = fuse_search(Sd, Sb_dev, "dev")
        out[mode] = dict(H7=e["H@7"], F1=e["F1@7"], r100=e["recall@100"], fused_H7=H, fused_F1=f1, alpha=a)
    return out, emb

def train_model(model, epochs, max_steps=None):
    opt = torch.optim.AdamW(param_groups(model), betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]
    spe = math.ceil(len(TR_PAIRS) / BATCH_PAIRS); total = epochs * spe
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, total - warm))))
    print(f"{spe} steps/epoch | total {total} steps | warmup {warm}")
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, step, done, t_train = [], 0, False, time.time()
    best = dict(score=None, epoch=0, state=None)
    for ep in range(1, epochs + 1):
        rng = random.Random(SEED + ep); pairs = TR_PAIRS[:]; rng.shuffle(pairs)
        model.train(); t0 = time.time(); L, A, Gp = [], [], []
        for b in range(0, len(pairs), BATCH_PAIRS):
            loss, st = step_loss(model, pairs[b:b + BATCH_PAIRS], rng)
            m = lr_mult(step)
            for g in opt.param_groups: g["lr"] = g["base_lr"] * m
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
            L.append(loss.item()); A.append(st["acc"]); Gp.append(st["gap"])
            if step == 10:
                per = (time.time() - t0) / 10
                print(f"  >> speed {per:.2f} s/step ({st['seqs']} seqs) | projected {per * spe / 60:.1f} min/epoch"
                      + (f" | peak GPU {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device.type == "cuda" else ""))
            if step % 50 == 0:
                print(f"  ep {ep} step {step}/{total} | loss {np.mean(L[-50:]):.4f} | in-batch acc {np.mean(A[-50:]):.3f} | "
                      f"cos(pos)-cos(hn) {np.mean(Gp[-50:]):+.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
            if max_steps and step >= max_steps: done = True; break
            if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping"); done = True; break
        ev, _ = dev_eval(model)
        score = max(ev[m]["fused_H7"] for m in ev) + 1e-3 * max(ev[m]["H7"] for m in ev)
        rec = dict(epoch=ep, loss=float(np.mean(L)), acc=float(np.mean(A)), gap=float(np.mean(Gp)), steps=step, secs=time.time() - t0, dev=ev)
        history.append(rec)
        print(f"epoch {ep}: loss {rec['loss']:.4f} acc {rec['acc']:.3f} | dev dense H@7 mean {ev['mean']['H7']} / maxsim {ev['maxsim']['H7']} | "
              f"fused H@7 mean {ev['mean']['fused_H7']} (a={ev['mean']['alpha']}) / maxsim {ev['maxsim']['fused_H7']} (a={ev['maxsim']['alpha']})")
        if best["score"] is None or score > best["score"]:
            best = dict(score=score, epoch=ep, state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
            torch.save(best, RUN_DIR / "best_trainable.pt")
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0)
    return history, best, stats

history, best, stats = train_model(model, EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)
print(f"\nbest epoch {best['epoch']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

train: 817 queries with gold | 5308 (query, positive) pairs | queries without a hard negative: 0
332 steps/epoch | total 1328 steps | warmup 66
  >> speed 4.20 s/step (152 seqs) | projected 23.2 min/epoch | peak GPU 3.0 GB
  ep 1 step 50/1328 | loss 2.2413 | in-batch acc 0.378 | cos(pos)-cos(hn) -0.0090 | lr x0.758 | 210s
  ep 1 step 100/1328 | loss 1.9951 | in-batch acc 0.430 | cos(pos)-cos(hn) -0.0010 | lr x0.998 | 421s
  ep 1 step 150/1328 | loss 1.9659 | in-batch acc 0.455 | cos(pos)-cos(hn) +0.0032 | lr x0.990 | 631s
  ep 1 step 200/1328 | loss 1.8127 | in-batch acc 0.456 | cos(pos)-cos(hn) +0.0090 | lr x0.974 | 841s
  ep 1 step 250/1328 | loss 1.9207 | in-batch acc 0.431 | cos(pos)-cos(hn) +0.0040 | lr x0.952 | 1051s
  ep 1 step 300/1328 | loss 1.9257 | in-batch acc 0.441 | cos(pos)-cos(hn) +0.0048 | lr x0.922 | 1262s
epoch 1: loss 1.9658 acc 0.435 | dev dense H@7 mean 290 / maxsim 296 | fused H@7 mean 400 (a=0.65) / maxsim 400 (a=0.35)
  ep 2 step 350/1328 | loss 1.8521 | in-bat

## Final dev evaluation and analysis
With the best epoch restored: dense-only and fused scores for both score modes (the better one becomes `SCORE_MODE`), the full `α` curve, reciprocal-rank fusion, the F1@K curve, the rank of every gold case under the fused score (versus under BM25 alone), how many gold cases the fused ranking **adds** to BM25's top 7 and how many it **loses**, and the per-query hit distribution.

In [20]:
# Cell 19 — Final dev evaluation: score mode, alpha curve, RRF, F1@K, gold ranks, gains vs BM25
ev_final, emb_dev = dev_eval(model)
if SCORE_MODE == "auto":
    SCORE_MODE = max(("mean", "maxsim"), key=lambda m: (ev_final[m]["fused_H7"], ev_final[m]["H7"]))
Sd_dev = self_mask(dense_scores(emb_dev, POOLING, SCORE_MODE), "dev")
(ALPHA, H_f, F1_f, r100_f), curve = fuse_search(Sd_dev, Sb_dev, "dev")
Sf_dev = fused(Sd_dev, Sb_dev, ALPHA)
e_d, e_f = evaluate(Sd_dev, "dev"), evaluate(Sf_dev, "dev", ks=tuple(range(1, 21)))
e_rrf = evaluate(rrf(Sd_dev, Sb_dev), "dev")
print(f"SCORE_MODE={SCORE_MODE} | POOLING={POOLING} | ALPHA={ALPHA}")
print(f"  BM25 only : H@7 {e_b['H@7']:4d} F1@7 {e_b['F1@7']:.4f} recall@100 {e_b['recall@100']:.3f} MRR {e_b['MRR']:.3f}")
print(f"  dense only: H@7 {e_d['H@7']:4d} F1@7 {e_d['F1@7']:.4f} recall@100 {e_d['recall@100']:.3f} MRR {e_d['MRR']:.3f}")
print(f"  fused     : H@7 {e_f['H@7']:4d} F1@7 {e_f['F1@7']:.4f} recall@100 {e_f['recall@100']:.3f} MRR {e_f['MRR']:.3f}  (P@7 {e_f['P@7']:.3f} R@7 {e_f['R@7']:.3f})")
print(f"  RRF       : H@7 {e_rrf['H@7']:4d} F1@7 {e_rrf['F1@7']:.4f} recall@100 {e_rrf['recall@100']:.3f}")
print("  alpha curve (alpha, H@7):", [(a, H) for a, H, _, _ in curve])
print("  F1@K (fused):", {k: round(e_f[f'F1@{k}'], 4) for k in range(1, 21)})

gcols = gold_cols("dev", Sd_dev.shape[0])
def ranks(S):
    o = np.argsort(-S, axis=1); r = np.empty_like(o); r[np.arange(len(o))[:, None], o] = np.arange(o.shape[1])
    return np.asarray([r[i, c] + 1 for i, g in enumerate(gcols) for c in g])
rb, rf, rd = ranks(Sb_dev), ranks(Sf_dev), ranks(Sd_dev)
print(f"  gold rank median: BM25 {np.median(rb):.0f} | dense {np.median(rd):.0f} | fused {np.median(rf):.0f}  "
      f"(top-100 share: {np.mean(rb <= 100):.3f} / {np.mean(rd <= 100):.3f} / {np.mean(rf <= 100):.3f})")
Tb, Tf = topk(Sb_dev, 7), topk(Sf_dev, 7)
gain = sum(len((set(Tf[i].tolist()) - set(Tb[i].tolist())) & gcols[i]) for i in range(len(gcols)))
loss_ = sum(len((set(Tb[i].tolist()) - set(Tf[i].tolist())) & gcols[i]) for i in range(len(gcols)))
hits = np.asarray([len(set(Tf[i].tolist()) & gcols[i]) for i in range(len(gcols))])
print(f"  vs BM25 top-7: gold cases gained {gain} | lost {loss_} | net {gain - loss_:+d}")
print(f"  hits@7 per query (fused): {dict(sorted(Counter(hits.tolist()).items()))} | zero-hit queries {(hits == 0).mean():.1%}")
FINAL = dict(score_mode=SCORE_MODE, pooling=POOLING, alpha=ALPHA, bm25=e_b, dense=e_d,
             fused={k: v for k, v in e_f.items() if k.endswith("@7") or k.startswith("recall") or k == "MRR"},
             f1_curve={k: e_f[f"F1@{k}"] for k in range(1, 21)}, rrf=e_rrf, alpha_curve=curve,
             gold_rank_median=dict(bm25=float(np.median(rb)), dense=float(np.median(rd)), fused=float(np.median(rf))),
             gained=gain, lost=loss_, zero_hit=float((hits == 0).mean()), by_epoch_modes=ev_final)

SCORE_MODE=mean | POOLING=wmean | ALPHA=0.25
  BM25 only : H@7  395 F1@7 0.4391 recall@100 0.754 MRR 0.895
  dense only: H@7  299 F1@7 0.3324 recall@100 0.718 MRR 0.702
  fused     : H@7  405 F1@7 0.4503 recall@100 0.768 MRR 0.899  (P@7 0.490 R@7 0.416)
  RRF       : H@7  369 F1@7 0.4102 recall@100 0.782
  alpha curve (alpha, H@7): [(0.0, 395), (0.05, 398), (0.1, 399), (0.15, 400), (0.2, 401), (0.25, 405), (0.3, 404), (0.35, 404), (0.4, 402), (0.45, 401), (0.5, 399), (0.55, 402), (0.6, 404), (0.65, 402), (0.7, 402), (0.75, 400), (0.8, 398), (0.85, 392), (0.9, 379), (0.95, 351), (1.0, 299)]
  F1@K (fused): {1: 0.1852, 2: 0.2928, 3: 0.3632, 4: 0.4042, 5: 0.4312, 6: 0.4426, 7: 0.4503, 8: 0.4476, 9: 0.4462, 10: 0.4357, 11: 0.4289, 12: 0.4203, 13: 0.4116, 14: 0.4038, 15: 0.3974, 16: 0.3894, 17: 0.3787, 18: 0.3668, 19: 0.3583, 20: 0.3498}
  gold rank median: BM25 12 | dense 21 | fused 11  (top-100 share: 0.754 / 0.718 / 0.768)
  vs BM25 top-7: gold cases gained 13 | lost 3 | net +10
  hits@7

In [21]:
# Cell 20 — Test: embed, fuse with BM25, prediction.json -> zip (full runs only)
def validate_prediction(pred, split="test"):
    pool_ids = {DOCS[k]["id"] for k in POOL[split]}; qids = [q["id"] for q in QRY[split]]
    assert set(pred) == set(qids), f"missing/extra queries: {len(set(qids) - set(pred))} missing, {len(set(pred) - set(qids))} extra"
    for qid, lst in pred.items():
        assert isinstance(lst, list) and len(lst) == K_EVAL and len(set(lst)) == K_EVAL, f"{qid}: need {K_EVAL} unique ids"
        assert all(isinstance(c, str) and c in pool_ids for c in lst), f"{qid}: id not in the {split} pool"
    return True

TEST = {}
if not SMOKE_TEST:
    t0 = time.time()
    emb_test = embed_split(model, "test", (POOLING,))
    Sd_test = self_mask(dense_scores(emb_test, POOLING, SCORE_MODE), "test")
    Sb_test = self_mask(bm25_matrix("test"), "test")
    Sf_test = fused(Sd_test, Sb_test, ALPHA)
    T7 = topk(Sf_test, K_EVAL)
    PRED = {q["id"]: [DOCS[POOL["test"][j]]["id"] for j in T7[i].tolist()] for i, q in enumerate(QRY["test"])}
    validate_prediction(PRED)
    sub_dir = RUN_DIR / "submission"; sub_dir.mkdir(exist_ok=True)
    (sub_dir / "prediction.json").write_text(json.dumps(PRED, indent=1))
    zpath = RUN_DIR / f"submission_{RUN_NAME}.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z: z.write(sub_dir / "prediction.json", arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z: assert z.namelist() == ["prediction.json"]
    Tb7 = topk(Sb_test, K_EVAL)
    overlap = np.mean([len(set(T7[i].tolist()) & set(Tb7[i].tolist())) / K_EVAL for i in range(len(T7))])
    TEST = dict(seconds=time.time() - t0, embed_seconds=emb_test["secs"], chunks=emb_test["n_chunks"], top7_overlap_with_bm25=float(overlap))
    print(f"test: {len(PRED)} queries -> {zpath} | embed {emb_test['secs']:.0f}s ({emb_test['n_chunks']} chunks) | "
          f"top-7 overlap with BM25 {overlap:.2f}")
    if SAVE_SCORES:
        def top_list(S, split):
            T = topk(S, SAVE_TOP)
            return {q["id"]: [[DOCS[POOL[split][j]]["id"], round(float(S[i, j]), 5)] for j in T[i].tolist() if np.isfinite(S[i, j])]
                    for i, q in enumerate(QRY[split][: S.shape[0]])}
        (RUN_DIR / "dense_scores.json").write_text(json.dumps(dict(run=RUN_NAME, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA,
                                                                   dev=top_list(Sd_dev, "dev"), test=top_list(Sd_test, "test"))))
        print("saved dense top-%d scores for dev/test" % SAVE_TOP)

test: 237 queries -> /kaggle/working/runs/p7_dec02_lorma/submission_p7_dec02_lorma.zip | embed 134s (18838 chunks) | top-7 overlap with BM25 0.97
saved dense top-300 scores for dev/test


## KD teacher export (for the distillation run)
For every train query: the teacher's **raw cosine scores** (chosen pooling and score mode) over its BM25 top-300 ∪ its gold set, keyed by the pool's ID strings, together with `τ`. The student run turns them into a listwise target `softmax(s/τ)` over the same candidate list (and pairwise margins), so it learns the teacher's *ranking*, not just the binary gold labels. Only cosine similarities are exported, so the student can be any architecture with any embedding size.

In [22]:
# Cell 21 — Save trainable weights + log; teacher export
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    if isinstance(o, Path): return str(o)
    return o
torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, RUN_DIR / "trainable_final.pt")
log = dict(run=RUN_NAME, method=METHOD, hp=HP[METHOD], view=dict(chunk=CHUNK_TOKENS, max_doc_chunks=MAX_DOC_CHUNKS, query_view=QUERY_VIEW,
           ctx_window=CTX_WINDOW, q_max_chunks=Q_MAX_CHUNKS), train=dict(epochs=EPOCHS, B=BATCH_PAIRS, C=C_TRAIN, QC=QC_TRAIN, hn_top=HN_TOP, tau=TAU),
           params_total=model.n_total, params_trainable=model.n_trainable, zero_shot=ZS, history=history, best_epoch=best["epoch"],
           stats=stats, final=FINAL, test=TEST, bm25_cfg=BM25_CFG)
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=1))

TEACHER = {}
if EXPORT_TEACHER and not SMOKE_TEST:
    t0 = time.time()
    emb_tr = embed_split(model, "train", (POOLING,))
    S_tr = self_mask(dense_scores(emb_tr, POOLING, SCORE_MODE), "train")
    col = {k: j for j, k in enumerate(POOL["train"])}
    exp = {}
    for i, q in enumerate(QRY["train"]):
        cands = [c for c, _ in BM25["train"][q["key"]]] + [g for g in (q["gold"] or [])]
        seen, row = set(), []
        for c in cands:
            if c in seen or c == q["key"] or c not in col: continue
            seen.add(c); row.append([DOCS[c]["id"], round(float(S_tr[i, col[c]]), 5)])
        exp[q["id"]] = row
    tpath = RUN_DIR / f"teacher_{RUN_NAME}.json"
    tpath.write_text(json.dumps(dict(run=RUN_NAME, tau=TAU, pooling=POOLING, mode=SCORE_MODE, alpha=ALPHA, scores=exp)))
    TEACHER = dict(seconds=time.time() - t0, queries=len(exp), mean_list=float(np.mean([len(v) for v in exp.values()])),
                   mb=tpath.stat().st_size / 1e6)
    print(f"teacher export: {TEACHER} -> {tpath}")
for f in sorted(RUN_DIR.iterdir()): print(f"   {f.name:40s} {f.stat().st_size / 1e6 if f.is_file() else 0:8.2f} MB")

teacher export: {'seconds': 326.9361424446106, 'queries': 827, 'mean_list': 300.9782345828295, 'mb': 6.2083} -> /kaggle/working/runs/p7_dec02_lorma/teacher_p7_dec02_lorma.json
   best_trainable.pt                            8.97 MB
   dense_scores.json                            2.66 MB
   run_log.json                                 0.01 MB
   submission                                   0.00 MB
   submission_p7_dec02_lorma.zip                0.01 MB
   teacher_p7_dec02_lorma.json                  6.21 MB
   trainable_final.pt                           8.97 MB


In [23]:
# Cell 22 — PASTE TO CLAUDE
def _r(o, n=4):
    if isinstance(o, dict): return {k: _r(v, n) for k, v in o.items()}
    if isinstance(o, list): return [_r(v, n) for v in o]
    if isinstance(o, float): return round(o, n)
    return o
short = dict(run=RUN_NAME, smoke=SMOKE_TEST, hp=HP[METHOD], params_trainable=model.n_trainable, params_total=model.n_total,
             zero_shot=ZS, history=[{k: v for k, v in h.items()} for h in history], best_epoch=best["epoch"], stats=stats,
             final={k: v for k, v in FINAL.items() if k not in ("alpha_curve",)}, alpha_curve=[(a, H) for a, H, _, _ in FINAL["alpha_curve"]],
             test=TEST, teacher=TEACHER)
print("================ PASTE TO CLAUDE ================")
print(json.dumps(_r(_jsonable(short)), separators=(",", ":")))
print("=================================================")

================ PASTE TO CLAUDE ================
{"run":"p7_dec02_lorma","smoke":false,"hp":{"lr":0.0005,"head_lr":0.0002,"wd":0.0,"r":16,"alpha":32,"variant":"plus"},"params_trainable":2228224,"params_total":57160192,"zero_shot":[{"pooling":"mean","mode":"mean","center":false,"H7":216,"F1":0.2401,"r100":0.6321,"MRR":0.5694,"fused_H7":397,"fused_F1":0.4414,"alpha":0.3},{"pooling":"mean","mode":"mean","center":true,"H7":221,"F1":0.2457,"r100":0.6372,"MRR":0.5994,"fused_H7":397,"fused_F1":0.4414,"alpha":0.1},{"pooling":"mean","mode":"maxsim","center":false,"H7":218,"F1":0.2424,"r100":0.6156,"MRR":0.6004,"fused_H7":398,"fused_F1":0.4425,"alpha":0.3},{"pooling":"mean","mode":"maxsim","center":true,"H7":223,"F1":0.2479,"r100":0.63,"MRR":0.6313,"fused_H7":398,"fused_F1":0.4425,"alpha":0.15},{"pooling":"wmean","mode":"mean","center":false,"H7":218,"F1":0.2424,"r100":0.6434,"MRR":0.5817,"fused_H7":397,"fused_F1":0.4414,"alpha":0.35},{"pooling":"wmean","mode":"mean","center":true,"H7":227,"F1"